# Big Data и ML — полный pipeline

PostgreSQL → MinIO → витрины → ML → Superset

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/etl')
sys.path.insert(0, '/home/jovyan/work/ml')

from export_to_minio import main as export_main
from process_marts import main as marts_main
from train_models import main as ml_main

export_main()
marts_main()
ml_main()

## Задание 1: аналитика добычи

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from config import pg_url

engine = create_engine(pg_url('postgres'))
prod = pd.read_sql('SELECT * FROM marts.mart_production ORDER BY prod_date', engine)
kpi = pd.read_sql('SELECT * FROM marts.mart_well_kpi ORDER BY avg_flow_tpd DESC', engine)
display(prod.head())
display(kpi.head())
print('Best well:', kpi.iloc[0]['well_id'])
print('Worst well:', kpi.iloc[-1]['well_id'])

## Задание 2: прогноз дебита (ML)

In [ ]:
import json
from pathlib import Path
metrics = json.loads(Path('/home/jovyan/work/ml/artifacts/metrics.json').read_text())
print(json.dumps(metrics['flow_forecast'], indent=2))
pred = pd.read_sql('SELECT * FROM marts.mart_flow_predictions LIMIT 10', engine)
pred.head()

## Задание 3: аномалии насосов

In [ ]:
fail = pd.read_sql("SELECT * FROM marts.mart_failure_predictions WHERE pump_id='P-W-003' ORDER BY ts", engine)
fail[['ts','vibration_mm_s','temperature_c','is_anomaly_iso','risk_score']].tail(10)

## Задание 4: логистика

In [ ]:
log = pd.read_sql('SELECT * FROM marts.mart_logistics', engine)
deliveries = pd.read_sql('SELECT weather, AVG(delay_hours) delay, AVG(cost_usd/distance_km) cost_km FROM raw.deliveries GROUP BY weather', engine)
display(log)
display(deliveries)